<a href="https://colab.research.google.com/github/intanoktavianipresia/Studi_Kasus_Deep_Learning/blob/main/Sesi_5_HCI_Visualisasi_Data_Interaktif.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ============================================================
# SESI 5: HUMAN-COMPUTER INTERACTION (HCI) & DATA VISUALIZATION
# Studi Kasus: Visualisasi Interaktif Harapan Hidup Antarnegara
# Dataset: Gapminder
# ============================================================

# SESI 1 - INSTALASI DAN IMPORT LIBRARY
!pip -q install plotly ipywidgets

import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import ipywidgets as widgets

print("Library berhasil disiapkan.")

# SESI 2 - MENGAMBIL DATASET
# Dataset Gapminder berisi data historis negara-negara.
df = px.data.gapminder()

print("\nDataset berhasil dimuat.")
print("Jumlah baris:", df.shape[0])
print("Jumlah kolom:", df.shape[1])
print("Rentang tahun:", df["year"].min(), "-", df["year"].max())
print("\nContoh data:")
display(df.head())

# SESI 3 - PEMERIKSAAN DAN PREPROCESSING DATA
print("\nInformasi dataset:")
df.info()

print("\nJumlah nilai kosong setiap kolom:")
display(df.isna().sum().to_frame("Jumlah Nilai Kosong"))

# Menghapus baris yang kosong pada kolom utama
kolom_penting = [
    "country", "continent", "year",
    "lifeExp", "pop", "gdpPercap"
]

df = df.dropna(subset=kolom_penting).copy()

# Memastikan tipe data sesuai
df["year"] = df["year"].astype(int)
df["lifeExp"] = pd.to_numeric(df["lifeExp"], errors="coerce")
df["pop"] = pd.to_numeric(df["pop"], errors="coerce")
df["gdpPercap"] = pd.to_numeric(df["gdpPercap"], errors="coerce")

df = df.dropna(subset=["lifeExp", "pop", "gdpPercap"])

print("\nJumlah data setelah preprocessing:", len(df))
print("Jumlah nilai kosong setelah preprocessing:",
      int(df[kolom_penting].isna().sum().sum()))

# SESI 4 - RINGKASAN DATA
tahun_terbaru = df["year"].max()
data_terbaru = df[df["year"] == tahun_terbaru].copy()

print(f"\nRingkasan data tahun {tahun_terbaru}:")
print("Jumlah negara:", data_terbaru["country"].nunique())
print("Rata-rata harapan hidup:",
      round(data_terbaru["lifeExp"].mean(), 2), "tahun")
print("Rata-rata populasi:",
      f"{data_terbaru['pop'].mean():,.0f}")
print("Rata-rata PDB per kapita:",
      f"${data_terbaru['gdpPercap'].mean():,.2f}")

# SESI 5 - VISUALISASI INTERAKTIF
# Grafik dapat digunakan untuk melihat hubungan PDB dan harapan hidup.
fig = px.scatter(
    data_terbaru,
    x="gdpPercap",
    y="lifeExp",
    size="pop",
    color="continent",
    hover_name="country",
    log_x=True,
    size_max=45,
    title=f"Hubungan PDB per Kapita dan Harapan Hidup ({tahun_terbaru})",
    labels={
        "gdpPercap": "PDB per Kapita (USD, nilai dataset)",
        "lifeExp": "Harapan Hidup (tahun)",
        "pop": "Populasi",
        "continent": "Benua"
    }
)

fig.update_layout(
    template="plotly_white",
    legend_title="Benua"
)
fig.show()

# SESI 6 - PERBANDINGAN ANTARBENUA
rata_benua = (
    data_terbaru.groupby("continent", as_index=False)
    .agg(
        Rata_Rata_Harapan_Hidup=("lifeExp", "mean"),
        Jumlah_Negara=("country", "nunique")
    )
    .sort_values("Rata_Rata_Harapan_Hidup", ascending=False)
)

print("\nRata-rata harapan hidup berdasarkan benua:")
display(rata_benua.round(2))

fig2 = px.bar(
    rata_benua,
    x="continent",
    y="Rata_Rata_Harapan_Hidup",
    text="Rata_Rata_Harapan_Hidup",
    title=f"Rata-rata Harapan Hidup Berdasarkan Benua ({tahun_terbaru})",
    labels={
        "continent": "Benua",
        "Rata_Rata_Harapan_Hidup": "Harapan Hidup (tahun)"
    }
)

fig2.update_traces(texttemplate="%{text:.1f}", textposition="outside")
fig2.update_layout(template="plotly_white")
fig2.show()

# SESI 7 - INTERAKSI PENGGUNA: PILIH TAHUN DAN BENUA
pilih_tahun = widgets.IntSlider(
    value=int(tahun_terbaru),
    min=int(df["year"].min()),
    max=int(df["year"].max()),
    step=5,
    description="Tahun:",
    continuous_update=False
)

pilih_benua = widgets.Dropdown(
    options=["Semua Benua"] + sorted(df["continent"].unique().tolist()),
    value="Semua Benua",
    description="Benua:"
)

area_output = widgets.Output()

def perbarui_visualisasi(tahun, benua):
    with area_output:
        clear_output(wait=True)

        data_filter = df[df["year"] == tahun].copy()

        if benua != "Semua Benua":
            data_filter = data_filter[data_filter["continent"] == benua]

        if data_filter.empty:
            print("Tidak ada data untuk pilihan tersebut.")
            return

        print("Jumlah negara:", data_filter["country"].nunique())
        print("Rata-rata harapan hidup:",
              round(data_filter["lifeExp"].mean(), 2), "tahun")

        grafik = px.scatter(
            data_filter,
            x="gdpPercap",
            y="lifeExp",
            size="pop",
            color="continent",
            hover_name="country",
            log_x=True,
            size_max=40,
            title=f"Visualisasi Data Tahun {tahun}",
            labels={
                "gdpPercap": "PDB per Kapita",
                "lifeExp": "Harapan Hidup (tahun)",
                "pop": "Populasi",
                "continent": "Benua"
            }
        )

        grafik.update_layout(template="plotly_white")
        grafik.show()

def saat_input_berubah(change):
    perbarui_visualisasi(pilih_tahun.value, pilih_benua.value)

pilih_tahun.observe(saat_input_berubah, names="value")
pilih_benua.observe(saat_input_berubah, names="value")

display(widgets.VBox([pilih_tahun, pilih_benua, area_output]))
perbarui_visualisasi(pilih_tahun.value, pilih_benua.value)

# SESI 8 - INSIGHT HASIL VISUALISASI
print("\nINSIGHT HASIL ANALISIS")

tertinggi = data_terbaru.loc[data_terbaru["lifeExp"].idxmax()]
terendah = data_terbaru.loc[data_terbaru["lifeExp"].idxmin()]

print(
    f"Negara dengan harapan hidup tertinggi pada {tahun_terbaru}: "
    f"{tertinggi['country']} ({tertinggi['lifeExp']:.2f} tahun)."
)
print(
    f"Negara dengan harapan hidup terendah pada {tahun_terbaru}: "
    f"{terendah['country']} ({terendah['lifeExp']:.2f} tahun)."
)
print(
    "Grafik interaktif membantu pengguna membandingkan negara, "
    "benua, populasi, PDB per kapita, dan harapan hidup."
)

# SESI 9 - KESIMPULAN
print("\n" + "=" * 55)
print("KESIMPULAN")
print("=" * 55)
print(
    "Studi kasus ini menerapkan prinsip HCI dan visualisasi data "
    "dengan menyediakan grafik interaktif, informasi saat kursor "
    "diarahkan ke titik, serta pilihan tahun dan benua."
)
print(
    "Interaksi tersebut memudahkan pengguna mengeksplorasi data "
    "dan membandingkan indikator antarnegara. Visualisasi menunjukkan "
    "pola dan hubungan, tetapi tidak membuktikan hubungan sebab-akibat."
)
print(
    "Sumber data: Gapminder melalui dataset bawaan Plotly Express."
)

Library berhasil disiapkan.

Dataset berhasil dimuat.
Jumlah baris: 1704
Jumlah kolom: 8
Rentang tahun: 1952 - 2007

Contoh data:


,country,continent,year,lifeExp,pop,gdpPercap,iso_alpha,iso_num
0,Afghanistan,Asia,1952,28.801,8425333,779.445314,AFG,4
1,Afghanistan,Asia,1957,30.332,9240934,820.853030,AFG,4
2,Afghanistan,Asia,1962,31.997,10267083,853.100710,AFG,4
3,Afghanistan,Asia,1967,34.020,11537966,836.197138,AFG,4
4,Afghanistan,Asia,1972,36.088,13079460,739.981106,AFG,4



Informasi dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1704 entries, 0 to 1703
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   country    1704 non-null   object 
 1   continent  1704 non-null   object 
 2   year       1704 non-null   int64  
 3   lifeExp    1704 non-null   float64
 4   pop        1704 non-null   int64  
 5   gdpPercap  1704 non-null   float64
 6   iso_alpha  1704 non-null   object 
 7   iso_num    1704 non-null   int64  
dtypes: float64(2), int64(3), object(3)
memory usage: 106.6+ KB

Jumlah nilai kosong setiap kolom:


,Jumlah Nilai Kosong
country,0
continent,0
year,0
lifeExp,0
pop,0
gdpPercap,0
iso_alpha,0
iso_num,0



Jumlah data setelah preprocessing: 1704
Jumlah nilai kosong setelah preprocessing: 0

Ringkasan data tahun 2007:
Jumlah negara: 142
Rata-rata harapan hidup: 67.01 tahun
Rata-rata populasi: 44,021,220
Rata-rata PDB per kapita: $11,680.07



Rata-rata harapan hidup berdasarkan benua:


,continent,Rata_Rata_Harapan_Hidup,Jumlah_Negara
4,Oceania,80.72,2
3,Europe,77.65,30
1,Americas,73.61,25
2,Asia,70.73,33
0,Africa,54.81,52



INSIGHT HASIL ANALISIS
Negara dengan harapan hidup tertinggi pada 2007: Japan (82.60 tahun).
Negara dengan harapan hidup terendah pada 2007: Swaziland (39.61 tahun).
Grafik interaktif membantu pengguna membandingkan negara, benua, populasi, PDB per kapita, dan harapan hidup.

KESIMPULAN
Studi kasus ini menerapkan prinsip HCI dan visualisasi data dengan menyediakan grafik interaktif, informasi saat kursor diarahkan ke titik, serta pilihan tahun dan benua.
Interaksi tersebut memudahkan pengguna mengeksplorasi data dan membandingkan indikator antarnegara. Visualisasi menunjukkan pola dan hubungan, tetapi tidak membuktikan hubungan sebab-akibat.
Sumber data: Gapminder melalui dataset bawaan Plotly Express.
